In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import polars as pl
from pathlib import Path
import numpy as np

from pudl.helpers import convert_cols_dtypes
from pudl.workspace.setup import PudlPaths
from pudl.dagster.build import build_interactive_defs
defs = build_interactive_defs(
    global_data_config_path="/Users/christinagosnell/code/pudl/src/pudl/package_data/settings/etl_full.yml"
)

pudl_paths = PudlPaths()

## Diff the outputs

In [ ]:
pks = ["plant_id_epa", "emissions_unit_id_epa"]
ca = (
    defs.load_asset_value("_out_epacems__yearly_operational_characteristics_CA")
    .to_pandas()
    .pipe(convert_cols_dtypes, name="out_epacems__yearly_operational_characteristics")
    .set_index(pks)
)
unrounded_cols = ["ramp_up_rate_fraction_of_max_gross_load_per_min","ramp_down_rate_fraction_of_max_gross_load_per_min"]
ca[unrounded_cols] = ca[unrounded_cols].round(2)
ca_og = (
    pd.read_csv(Path().cwd()/ "epa_op_char_output_df.csv")
    .pipe(convert_cols_dtypes, name="out_epacems__yearly_operational_characteristics")
    .set_index(pks)
    .rename(columns={'min_up_time_hr': 'min_up_time_hours', 'min_down_time_hr': "min_down_time_hours"})
)

In [ ]:
pks = ["plant_id_epa", "emissions_unit_id_epa"]
diff = pd.merge(
    left=ca,
    right=ca_og,
    right_index=True,
    left_index=True,
    how="outer",
    validate="1:1",
    suffixes=("_new","_jax"),
    indicator=True
).reset_index().set_index([c for c in pks if c != "ending_balance"]).sort_index()
diff._merge = diff._merge.cat.rename_categories({"right_only": "jax_only", "left_only": "new_only"})

assert diff[diff._merge != "both"].empty

In [ ]:
def compare_values_in_col(diff, compare_col):
    return (
        diff[
            ~np.isclose(
                diff[f"{compare_col}_new"],diff[f"{compare_col}_jax"],
                rtol=1e-2
            )
            & ~(diff[f"{compare_col}_new"].isnull() & diff[f"{compare_col}_jax"].isnull())
        ]
        .filter(regex=rf"(^{compare_col})_(new|jax)")
    )
for compare_col in ca:
    if compare_col in ["plant_id_eia", "state", "report_year"]:
        continue
    else:
        out = compare_values_in_col(diff, compare_col)[f"{compare_col}_new"].value_counts(dropna=False)
        if not out.empty:
            out_fr= out
            print(f"{compare_col}:    {out_fr.values.sum()}")

In [ ]:
# change the column name here to explore different columns
compare_values_in_col(diff, "min_stable_level")

In [ ]:
compare_col =  "min_down_time_hours"
diff.filter(regex=rf"(^{compare_col})_(new|jax)").describe()